In [ ]:
!pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
model_id = "deepvk/RuModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id, revision="patched-tokenizer")

model = AutoModelForMaskedLM.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model = model.eval()

text = "Байден безнадежный бездарь! Трамп превосходен [MASK]!"
inputs = tokenizer(text, return_tensors="pt").to(device)

masked_index = inputs["input_ids"][0].tolist().index(tokenizer.mask_token_id)

with torch.no_grad():
    outputs = model(**inputs)

predicted_token_id = outputs.logits[0, masked_index].argmax(axis=-1)
predicted_token = tokenizer.decode(predicted_token_id)

print(f"Input text: {text}")
print(f"Predicted token: {predicted_token}")


Input text: Байден безнадежный бездарь! Трамп превосходен [MASK]!
Predicted token:  всегда


In [ ]:
# После строки outputs = model(**inputs)

# Получаем топ-5 предсказаний
top_k = 5
logits = outputs.logits[0, masked_index]
top_k_ids = torch.topk(logits, k=top_k).indices

print(f"\nInput text: {text}")
print("Топ-5 предсказаний:")
for i, token_id in enumerate(top_k_ids, 1):
    token = tokenizer.decode(token_id)
    prob = torch.softmax(logits, dim=0)[token_id].item() * 100
    print(f"{i}. {token:<15} ({prob:.1f}%)")


Input text: Байден безнадежный бездарь! Трамп превосходен [MASK]!
Топ-5 предсказаний:
1.  всегда         (10.2%)
2.  всем           (4.5%)
3.  везде          (4.0%)
4.  бизнесмен      (2.3%)
5.  актер          (1.7%)
